# Multi-index RAG pipeline

A retrieval-augmented generation (RAG) pipeline that combines **semantic** (vector) and **lexical** (BM25) search over a document chunked by markdown sections.
The notebook builds the two indexes, wraps them in a `Retriever`, runs one search, and then closes the loop with a Claude call that uses the retrieved chunks as context.

## Pipeline

1. **Chunk** — `chunk_by_section` splits `report.md` at `## ` headers (one chunk per section).
   Alternative strategies (`chunk_by_char`, `chunk_by_sentence`) are defined for reference but not used by the pipeline.
2. **Embed** — `generate_embedding` calls VoyageAI (`voyage-3-large`) to turn each chunk into a vector.
3. **Index** — two indexes side by side:
   - `VectorIndex` stores chunks + embeddings; `search()` ranks by **cosine distance** between the query embedding and stored embeddings.
   - `BM25Index` stores chunks + token stats; `search()` ranks by **BM25** score (TF × IDF with length normalization).
4. **Retriever** — wraps both indexes. On `search()` it queries each, then fuses the rankings with **Reciprocal Rank Fusion** (RRF): a document's final score is the sum of `1 / (k_rrf + rank)` across indexes.
5. **Generate** — the final cell feeds the top chunks to Claude as **system** context and asks the question. Context goes in `system` (not `user`) so the stable prefix is cacheable across follow-up questions on the same document.

## Why hybrid

- **Semantic (vector) search** excels at paraphrase and conceptual similarity.
  Weak on opaque identifiers (`INC-2023-Q4-011`) because the embedding model has no semantic signal for them — it tokenizes the code into generic sub-pieces (digits, hyphens) that do not point anywhere specific.
- **Lexical (BM25) search** excels at rare exact-term matching: rare tokens have high IDF, so `INC-2023-Q4-011` dominates the score and the chunks that contain it bubble up.
  Weak on paraphrase: a query with no surface-token overlap scores zero.
- Combined with RRF: a document relevant to either index ranks well overall, so neither failure mode kills the result.

## Models and dependencies

- Embedding model: `voyage-3-large` (VoyageAI).
  Requires `VOYAGE_API_KEY` in `.env` and `voyageai>=0.5.0` in `pyproject.toml`.
- Generation model: `claude-haiku-4-5` (default from `chat`).


In [1]:
# Client Setup
import voyageai
from dotenv import load_dotenv

load_dotenv()

client = voyageai.Client()

/home/rturrado/Projects/anthropic/building-with-the-claude-api/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Chunk by section
import re


def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0
    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)
    return chunks


def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    start_idx = 0
    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        chunks.append(" ".join(sentences[start_idx:end_idx]))
        start_idx += max_sentences_per_chunk - overlap_sentences
        if start_idx < 0:
            start_idx = 0
    return chunks


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [3]:
# Embedding Generation
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    chunks = chunks if is_list else [chunks]
    result = client.embed(chunks, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [4]:
# VectorIndex implementation
import math
from typing import Any


def _dot_product(vec1: list[float], vec2: list[float]) -> float:
    if len(vec1) != len(vec2):
        raise ValueError("Vectors must have the same dimension")
    return sum(p * q for p, q in zip(vec1, vec2, strict=True))


def _euclidean_distance(vec1: list[float], vec2: list[float]) -> float:
    if len(vec1) != len(vec2):
        raise ValueError("Vectors must have the same dimension")
    return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2, strict=True)))


def _magnitude(vec: list[float]) -> float:
    return math.sqrt(sum(x * x for x in vec))


def _cosine_distance(vec1: list[float], vec2: list[float]) -> float:
    if len(vec1) != len(vec2):
        raise ValueError("Vectors must have the same dimension")

    mag1 = _magnitude(vec1)
    mag2 = _magnitude(vec2)

    if mag1 == 0 and mag2 == 0:
        return 0.0
    elif mag1 == 0 or mag2 == 0:
        return 1.0

    dot_prod = _dot_product(vec1, vec2)
    cosine_similarity = dot_prod / (mag1 * mag2)
    cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

    return 1.0 - cosine_similarity


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: list[list[float]] = []
        self.documents: list[dict[str, Any]] = []
        self._vector_dim: int | None = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def add_documents(self, documents: list[dict[str, Any]]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")

        if not isinstance(documents, list):
            raise TypeError("Documents must be a list of dictionaries.")

        if not documents:
            return

        contents = []
        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"Document at index {i} must be a dictionary.")
            if "content" not in doc:
                raise ValueError(f"Document at index {i} must contain a 'content' key.")
            if not isinstance(doc["content"], str):
                raise TypeError(f"Document 'content' at index {i} must be a string.")
            contents.append(doc["content"])

        vectors = self._embedding_fn(contents)

        for vector, document in zip(vectors, documents, strict=True):
            self.add_vector(vector=vector, document=document)

    def search(self, query: Any, k: int = 1) -> list[tuple[dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError("Embedding function not provided for string query.")
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(isinstance(x, (int, float)) for x in query):
            query_vector = query
        else:
            raise TypeError("Query must be either a string or a list of numbers.")

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        dist_func = _cosine_distance if self._distance_metric == "cosine" else _euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: dict[str, Any]):
        if not isinstance(vector, list) or not all(isinstance(x, (int, float)) for x in vector):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}")

        self.vectors.append(list(vector))
        self.documents.append(document)

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        count = len(self)
        dim = self._vector_dim
        metric = self._distance_metric
        has_embedding_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={count}, dim={dim}, metric='{metric}', has_embedding_fn='{has_embedding_fn}')"

In [5]:
# BM25 implementation
from collections import Counter
from collections.abc import Callable


class BM25Index:
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Callable[[str], list[str]] | None = None,
    ):
        self.documents: list[dict[str, Any]] = []
        self._corpus_tokens: list[list[str]] = []
        self._doc_len: list[int] = []
        self._doc_freqs: dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> list[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: list[str]):
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: dict[str, Any]):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def add_documents(self, documents: list[dict[str, Any]]):
        if not isinstance(documents, list):
            raise TypeError("Documents must be a list of dictionaries.")

        if not documents:
            return

        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"Document at index {i} must be a dictionary.")
            if "content" not in doc:
                raise ValueError(f"Document at index {i} must contain a 'content' key.")
            if not isinstance(doc["content"], str):
                raise TypeError(f"Document 'content' at index {i} must be a string.")

            content = doc["content"]
            doc_tokens = self._tokenizer(content)

            self.documents.append(doc)
            self._corpus_tokens.append(doc_tokens)
            self._update_stats_add(doc_tokens)

        self._index_built = False

    def _compute_bm25_score(self, query_tokens: list[str], doc_index: int) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (1 - self.b + self.b * (doc_length / self._avg_doc_len))
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query: Any,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> list[tuple[dict[str, Any], float]]:
        if not self.documents:
            return []

        if isinstance(query, str):
            query_text = query
        else:
            raise TypeError("Query must be a string for BM25Index.")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25VectorStore(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"

In [6]:
# Retriever implementation
from typing import Protocol


class SearchIndex(Protocol):
    def add_document(self, document: dict[str, Any]) -> None: ...

    # Added the 'add_documents' method to avoid rate limiting errors from VoyageAI
    def add_documents(self, documents: list[dict[str, Any]]) -> None: ...

    def search(self, query: Any, k: int = 1) -> list[tuple[dict[str, Any], float]]: ...


class Retriever:
    def __init__(self, *indexes: SearchIndex):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    def add_document(self, document: dict[str, Any]):
        for index in self._indexes:
            index.add_document(document)

    # Added the 'add_documents' method to avoid rate limiting errors from VoyageAI
    def add_documents(self, documents: list[dict[str, Any]]):
        for index in self._indexes:
            index.add_documents(documents)

    def search(self, query_text: str, k: int = 1, k_rrf: int = 60) -> list[tuple[dict[str, Any], float]]:
        if not isinstance(query_text, str):
            raise TypeError("Query text must be a string.")
        if k <= 0:
            raise ValueError("k must be a positive integer.")
        if k_rrf < 0:
            raise ValueError("k_rrf must be non-negative.")

        all_results = [index.search(query_text, k=k * 5) for index in self._indexes]

        doc_ranks = {}
        for idx, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                doc_id = id(doc)
                if doc_id not in doc_ranks:
                    doc_ranks[doc_id] = {
                        "doc_obj": doc,
                        "ranks": [float("inf")] * len(self._indexes),
                    }
                doc_ranks[doc_id]["ranks"][idx] = rank + 1

        def calc_rrf_score(ranks: list[float]) -> float:
            return sum(1.0 / (k_rrf + r) for r in ranks if r != float("inf"))

        scored_docs: list[tuple[dict[str, Any], float]] = [
            (ranks["doc_obj"], calc_rrf_score(ranks["ranks"])) for ranks in doc_ranks.values()
        ]

        filtered_docs = [(doc, score) for doc, score in scored_docs if score > 0]
        filtered_docs.sort(key=lambda x: x[1], reverse=True)

        return filtered_docs[:k]

In [7]:
# Chunk source text by section
with open("./report.md") as f:
    text = f.read()

chunks = chunk_by_section(text)

In [8]:
# Create a vector index, a bm25 index, then use them to create a Retriever
vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index = BM25Index()

retriever = Retriever(bm25_index, vector_index)

In [9]:
# Add all chunks to the retriever, which internally passes them along to both indexes
# Note: converted to a bulk operation to avoid rate limiting errors from VoyageAI
retriever.add_documents([{"content": chunk} for chunk in chunks])

In [10]:
prompt = "What happened with INC-2023-Q4-011?"
results = retriever.search(prompt, 3)

for doc, score in results:
    print(score, "\n", doc["content"][0:200], "\n---\n")

0.03252247488101534 
 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems 
---

0.03252247488101534 
 Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center successfully contained and remediated a targeted intrusion attempt tracked as `INC-2 
---

0.031024531024531024 
 Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of findings drawn from standard departmental reporting cycles, specialized project updates 
---



In [11]:
# Close the RAG loop: feed the retrieved chunks to Claude as system context, then ask the question.
# Context goes in `system` (not `user`) for prompt-cache friendliness: the system prefix stays stable
# across follow-up questions on the same document and gets cached by the API.
from chat import add_user_message, chat

context = "\n\n".join(doc["content"] for doc, _ in results)
system = f"Answer based only on the following context:\n\n{context}"

messages = []
add_user_message(messages, prompt)
answer = chat(messages, system=system)
print(answer)

# INC-2023-Q4-011 Summary

Based on the provided context, INC-2023-Q4-011 refers to **two related but distinct issues**:

## Cybersecurity Incident (Section 10)
A targeted intrusion attempt was successfully contained and remediated. Key details include:

- **Threat Actor**: ShadowNet Syndicate
- **Initial Access**: Spear-phishing email targeting finance department personnel
- **Attack Details**: 
  - Anomalous process execution (PID: 7812) detected on workstation WS-FIN-112
  - Malware attempted lateral movement toward server SRV-FIN-03
- **Response Actions**:
  - Isolated affected systems
  - Blocked command-and-control infrastructure (IP: 198.51.100.24)
  - Deployed updated endpoint policies
  - Implemented enhanced perimeter filtering (Firewall Rule ID: FN7832)
- **Status**: Containment completed; forensics analysis ongoing

## Software Engineering Response (Section 2)
The Software Engineering division provided assistance in response to this incident during Q4 2024, focusing on stab